In [11]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import cv2
import glob
from transformers import BlipForConditionalGeneration, AutoProcessor

# --- Configuration ---
BASE_PATH = "/Users/ayraj/Desktop/video_captioning"
MODEL_DIR = os.path.join(BASE_PATH, "blip_video_model_2")
VIDEO_FOLDER = os.path.join(BASE_PATH, "TrainValVideo")
OUTPUT_FOLDER = os.path.join(BASE_PATH, "attention_heatmaps")
NUM_FRAMES = 8

def process_batch():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    video_files = glob.glob(os.path.join(VIDEO_FOLDER, "*.mp4"))[:20]
    
    if not video_files:
        print(f"❌ ERROR: No .mp4 videos found in {VIDEO_FOLDER}")
        return

    print("Loading model and processor...")
    processor = AutoProcessor.from_pretrained(MODEL_DIR)
    model = BlipForConditionalGeneration.from_pretrained(MODEL_DIR)
    model.eval()

    for idx, video_path in enumerate(video_files):
        vid_name = os.path.basename(video_path)
        print(f"\n--- [{idx+1}/20] Processing {vid_name} ---")
        
        # 1. Capture 8 frames
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            print(f"⚠️ Skipping {vid_name}: Unreadable frame count.")
            continue
            
        indices = np.linspace(0, total_frames - 1, NUM_FRAMES, dtype=int)
        frames = []
        for i in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (224, 224))
                frames.append(frame)
        cap.release()

        while len(frames) < NUM_FRAMES: frames.append(frames[-1])

        # 2. Preprocess & Generate
        inputs = processor(images=frames, return_tensors="pt")
        output_ids = model.generate(**inputs, max_length=20, num_beams=1)
        
        # 3. Extract Attention
        with torch.no_grad():
            vision_outputs = model.vision_model(pixel_values=inputs.pixel_values)
            decoder_outputs = model.text_decoder(
                input_ids=output_ids,
                encoder_hidden_states=vision_outputs[0],
                output_attentions=True,
                return_dict=True
            )

        cross_attentions = decoder_outputs.cross_attentions 
        if cross_attentions is None:
            continue

        # 4. Clean tokens
        raw_tokens = processor.tokenizer.convert_ids_to_tokens(output_ids[0])
        words, valid_indices = [], []
        for i, t in enumerate(raw_tokens):
            if t is not None:
                clean_t = t.replace('Ġ', '').replace('##', '')
                if clean_t not in ['[PAD]', '[CLS]', '[SEP]', '[EOS]', '']:
                    words.append(clean_t)
                    valid_indices.append(i)

        if not words: continue

        # 5. THE FIX: Safe Alignment
        attn_matrix = np.zeros((len(words), NUM_FRAMES))
        last_layer_attn = cross_attentions[-1]
        avg_attn = last_layer_attn.mean(dim=1).squeeze(0) 
        
        # Calculate offset in case the model dropped the hidden start token
        offset = len(raw_tokens) - avg_attn.shape[0]

        for matrix_row, original_idx in enumerate(valid_indices):
            # Align the index safely and clamp it so it can NEVER go out of bounds
            attn_idx = original_idx - offset
            attn_idx = max(0, min(attn_idx, avg_attn.shape[0] - 1))
            
            token_attn = avg_attn[attn_idx].detach().cpu().numpy()
            frame_chunks = np.array_split(token_attn, NUM_FRAMES)
            attn_matrix[matrix_row] = [chunk.mean() for chunk in frame_chunks]

        # 6. Save Visualization
        plt.figure(figsize=(10, 6))
        sns.heatmap(
            attn_matrix, annot=True, fmt=".3f", 
            xticklabels=[f"Frame {k+1}" for k in range(NUM_FRAMES)], 
            yticklabels=words, cmap="rocket_r"
        )
        plt.title(f"Temporal Attention Heatmap: {vid_name}")
        plt.xlabel("Timeline (Frames 1-8)")
        plt.ylabel("Generated Words")
        
        save_path = os.path.join(OUTPUT_FOLDER, f"heatmap_{vid_name}.png")
        plt.savefig(save_path, bbox_inches='tight')
        plt.close() 
        print(f"✅ Saved: heatmap_{vid_name}.png")

    print(f"\n🎉 All done! Check the '{OUTPUT_FOLDER}' folder for your 20 heatmaps.")

if __name__ == "__main__":
    process_batch()

Loading model and processor...


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 19936.15it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



--- [1/20] Processing video12.mp4 ---
✅ Saved: heatmap_video12.mp4.png

--- [2/20] Processing video6110.mp4 ---
✅ Saved: heatmap_video6110.mp4.png

--- [3/20] Processing video2376.mp4 ---
✅ Saved: heatmap_video2376.mp4.png

--- [4/20] Processing video938.mp4 ---
✅ Saved: heatmap_video938.mp4.png

--- [5/20] Processing video5419.mp4 ---
✅ Saved: heatmap_video5419.mp4.png

--- [6/20] Processing video4707.mp4 ---
✅ Saved: heatmap_video4707.mp4.png

--- [7/20] Processing video3068.mp4 ---
✅ Saved: heatmap_video3068.mp4.png

--- [8/20] Processing video4061.mp4 ---
✅ Saved: heatmap_video4061.mp4.png

--- [9/20] Processing video2410.mp4 ---
✅ Saved: heatmap_video2410.mp4.png

--- [10/20] Processing video1119.mp4 ---
✅ Saved: heatmap_video1119.mp4.png

--- [11/20] Processing video6676.mp4 ---
✅ Saved: heatmap_video6676.mp4.png

--- [12/20] Processing video6662.mp4 ---
✅ Saved: heatmap_video6662.mp4.png

--- [13/20] Processing video4075.mp4 ---
✅ Saved: heatmap_video4075.mp4.png

--- [14/20] P